# SLR SLR: add Products-6K to the detector on Colab

Products-6K (zenodo.org/records/4428917) is a grocery dataset from a Greek supermarket, made to help
visually impaired shoppers. It has no bounding boxes, so `training/get_products6k_data.py` maps each
product to one of our 25 classes and draws the boxes automatically (see that file for details).
This notebook then fine-tunes the Stage A model on Freiburg + Products-6K together.

1. **Runtime -> Change runtime type -> GPU** (T4 is fine).
2. Run the cells top to bottom. Downloads ~3.2 GB in total; converting takes a few minutes;
   training 60 epochs takes roughly 30-60 min on a T4.

License: Products-6K is CC BY-NC-ND 4.0. Fine for this class project; do not share the converted
images (they stay in `data/`, which is gitignored).

### How to read this notebook

Each gray box is a *cell*: click it and press **Shift+Enter** to run it. Lines starting with `#`
are comments, `!` runs a terminal command, `%` is a notebook command.

- **Products-6K candidate set**: studio photos of 500 products on white backgrounds (boxes are exact).
- **Products-6K query set**: phone photos of 104 products held in a hand inside the store
  (boxes come from Grounding DINO, an AI that finds objects described in words).
- **Review sheets**: thumbnails with the automatic boxes drawn on them, least certain first.

## 1. Get the code

Same as `train_colab.ipynb`: paste a GitHub token when asked, or upload a zip of the repo and set
`USE_ZIP = True`. Commit and push `training/get_products6k_data.py` and
`training/products6k_map.yaml` first, or use the zip option.

In [ ]:
# --- Get a copy of the project onto this Colab machine ---------------------------
# Colab is a free Google service that runs Python notebooks on a borrowed computer with
# a GPU. Each 'cell' below runs when you press Shift+Enter. Lines starting with ! are
# shell (terminal) commands, lines starting with % are notebook commands; everything
# else is ordinary Python.

# os = file/folder helpers, getpass = ask for a password without echoing it,
# subprocess = run another program (git) from Python.
import os, getpass, subprocess
# Settings you may change: USE_ZIP = True if you uploaded the repo as a zip file instead
# of giving a GitHub token; ZIP_NAME = the uploaded file's name; REPO = the GitHub repo.
USE_ZIP = False
ZIP_NAME = "VisuallyImpairedBodycam.zip"
REPO = "Ore-Porat/VisuallyImpairedBodycam"

# %cd changes the notebook's working folder (/content is Colab's scratch area).
%cd /content
# Option 1: the repo was uploaded as a zip. Unpack it quietly (-q) and overwrite (-o).
if USE_ZIP:
    !unzip -q -o {ZIP_NAME} -d /content/repo_zip
# os.walk lists every folder; next(...) grabs the first one that has a training/ subfolder.
    # the zip may contain a top-level folder; find the one with training/
    root = next(r for r, d, f in os.walk("/content/repo_zip") if "training" in d)
# Option 2: clone from GitHub, but only if the folder does not already exist (so re-running
# the cell is harmless). The token is typed in a hidden prompt and never stored.
    !rm -rf /content/VisuallyImpairedBodycam && cp -r "{root}" /content/VisuallyImpairedBodycam
# --depth 1 downloads only the latest version, not the whole history (faster).
elif not os.path.isdir("/content/VisuallyImpairedBodycam"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1",
# `del token` forgets the token variable as soon as it is no longer needed.
                    f"https://{token}@github.com/{REPO}.git", "/content/VisuallyImpairedBodycam"], check=True)
    del token
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", "/content/VisuallyImpairedBodycam", "remote", "set-url", "origin",
# Finally move into the repo folder so the following cells can run its scripts.
                    f"https://github.com/{REPO}.git"], check=True)
%cd /content/VisuallyImpairedBodycam

## 2. Install

In [ ]:
# ultralytics = YOLO, albumentations = blur augmentation, openpyxl = reads the .xlsx labels,
# transformers = Grounding DINO for the in-store photos. Colab already has torch.
!pip install -q "ultralytics>=8.3" albumentations pyyaml openpyxl transformers
# A tool to unpack .rar archives (Colab usually has one; install unrar if not).
!which unrar > /dev/null || which 7z > /dev/null || apt-get -qq install -y unrar > /dev/null
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Freiburg / GrocerEye data (the Stage A dataset)

Training on both datasets together keeps the model from forgetting what it already knows.

In [ ]:
!python training/get_grocereye_data.py

## 4. Products-6K: check the category mapping, then convert

`--inspect` prints every category and the class it maps to (or "no match"). Edit
`training/products6k_map.yaml` if something looks wrong, then run the conversion cell.

In [ ]:
# Downloads Candidate_Set.rar (110 MB) and Query_Set.rar (2.7 GB) into data/downloads/ the first time.
!python training/get_products6k_data.py --download all --inspect

In [ ]:
# Convert: map classes, draw boxes (plain background or Grounding DINO on the GPU), split by product.
!python training/get_products6k_data.py --download all --device cuda

## 5. Spot-check the automatic boxes

The least certain boxes come first (orange = Grounding DINO, green = plain background). If one is
wrong, open `data/products6k/boxes.json`, set `"skip": true` for that photo (or type a corrected
`"box"` with `"method": "manual"`), and rerun the conversion cell.

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("data/products6k/review/sheet_*.jpg"))[:3]:
    print(path)
    display(Image(filename=path))

## 6. Train on Freiburg + Products-6K, starting from the Stage A weights

In [ ]:
NAME = "grocereye_p6k"
EPOCHS = 60
!python training/train.py --name {NAME} --model models/grocereye_a/best.pt --data data/grocereye/data.yaml --data data/products6k/data.yaml --epochs {EPOCHS} --imgsz 320 --batch 64 --device 0 --workers 2

## 7. Evaluate on both test splits

The Freiburg score should stay about where Stage A was (0.93 mAP50); the Products-6K score shows
how well the model handles new brands and the hand-held store photos.

In [ ]:
!python training/evaluate.py --weights models/{NAME}/best.pt --split test --device 0
!python training/evaluate.py --weights models/{NAME}/best.pt --data data/products6k/data.yaml --split test --device 0 --name {NAME}_on_products6k --models-dir runs/eval_copies
from IPython.display import Markdown, display
display(Markdown(open(f"models/{NAME}/eval_report.md").read()))

## 8. Download the weights and reports

Put `best.pt` in `models/<NAME>/` on the laptop, then set `model.weights: models/<NAME>/best.pt` in
`config/server.yaml` (classes and `imgsz: 320` stay the same).

In [ ]:
!cd models && zip -q -r /content/{NAME}.zip {NAME}
!cp reports/eval_{NAME}_*.md /content/ 2>/dev/null || true
from google.colab import files
files.download(f"/content/{NAME}.zip")